setup

Defines the volume folders and the 4 tables we’re working on.

In [0]:
from pyspark.sql import functions as F

catalog = "workspace"
schema = "de_ecommerce"
base = "/Volumes/workspace/de_ecommerce/raw_files"
holding = f"{base}/holding"
incoming = f"{base}/incoming"
ckpt = f"{base}/_checkpoints"

tables = ["orders", "order_items", "order_payments", "order_reviews"]

reset

Empties the 4 bronze order tables (the tables and their columns stay), and clears the incoming and checkpoint folders so this notebook can be re-run from scratch.

In [0]:
for t in tables:
    spark.sql(f"TRUNCATE TABLE {catalog}.{schema}.bronze_{t}")

dbutils.fs.rm(incoming, True)
dbutils.fs.rm(ckpt, True)

for t in tables:
    print(t, spark.table(f"{catalog}.{schema}.bronze_{t}").count())

orders 0
order_items 0
order_payments 0
order_reviews 0


deliver the initial files

Copies each table’s initial folder from holding into incoming, like a file arriving from a source system. It then lists the files for orders.

In [0]:
for t in tables:
    dbutils.fs.cp(f"{holding}/{t}/initial", f"{incoming}/{t}/initial", recurse=True)

display(dbutils.fs.ls(f"{incoming}/orders/initial"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/de_ecommerce/raw_files/incoming/orders/initial/_SUCCESS,_SUCCESS,0,1793135385000
dbfs:/Volumes/workspace/de_ecommerce/raw_files/incoming/orders/initial/_committed_7866998030968877259,_committed_7866998030968877259,124,1793135385000
dbfs:/Volumes/workspace/de_ecommerce/raw_files/incoming/orders/initial/_started_7866998030968877259,_started_7866998030968877259,0,1793135386000
dbfs:/Volumes/workspace/de_ecommerce/raw_files/incoming/orders/initial/part-00000-tid-7866998030968877259-cbfc3878-f618-4262-97da-92ec48e105d0-189-1.c000.snappy.parquet,part-00000-tid-7866998030968877259-cbfc3878-f618-4262-97da-92ec48e105d0-189-1.c000.snappy.parquet,4552812,1793135386000


run Auto Loader

Defines a function ingest(table) that:

Watches incoming/<table>/ for files it hasn’t processed.
Adds the same two tracking columns as before: source_file (which file the row came from) and ingested_at (when it was loaded).
Lines up the columns with the existing bronze table.
Appends the new rows to that bronze table, then stops.

It then runs for all 4 tables and prints the row counts.

In [0]:
def ingest(t):
    target = f"{catalog}.{schema}.bronze_{t}"
    cols = spark.table(target).columns
    (spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", f"{ckpt}/{t}/schema")
        .load(f"{incoming}/{t}/")
        .withColumn("source_file", F.col("_metadata.file_path"))
        .withColumn("ingested_at", F.current_timestamp())
        .drop("_rescued_data")
        .select(*cols)
        .writeStream
        .option("checkpointLocation", f"{ckpt}/{t}/checkpoint")
        .trigger(availableNow=True)
        .toTable(target)
        .awaitTermination())
    print(t, "->", spark.table(target).count(), "rows")

for t in tables:
    ingest(t)

orders -> 45430 rows
order_items -> 51234 rows
order_payments -> 47871 rows
order_reviews -> 45370 rows


look at what was loaded

Shows which file the bronze orders rows came from, and 5 sample rows.

In [0]:
b = spark.table(f"{catalog}.{schema}.bronze_orders")
display(b.groupBy("source_file").count())
display(b.limit(5))

source_file,count
/Volumes/workspace/de_ecommerce/raw_files/incoming/orders/initial/part-00000-tid-7866998030968877259-cbfc3878-f618-4262-97da-92ec48e105d0-189-1.c000.snappy.parquet,45430


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,source_file,ingested_at
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,/Volumes/workspace/de_ecommerce/raw_files/incoming/orders/initial/part-00000-tid-7866998030968877259-cbfc3878-f618-4262-97da-92ec48e105d0-189-1.c000.snappy.parquet,2026-10-27T21:11:02.910Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z,/Volumes/workspace/de_ecommerce/raw_files/incoming/orders/initial/part-00000-tid-7866998030968877259-cbfc3878-f618-4262-97da-92ec48e105d0-189-1.c000.snappy.parquet,2026-10-27T21:11:02.910Z
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z,/Volumes/workspace/de_ecommerce/raw_files/incoming/orders/initial/part-00000-tid-7866998030968877259-cbfc3878-f618-4262-97da-92ec48e105d0-189-1.c000.snappy.parquet,2026-10-27T21:11:02.910Z
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09T00:00:00.000Z,/Volumes/workspace/de_ecommerce/raw_files/incoming/orders/initial/part-00000-tid-7866998030968877259-cbfc3878-f618-4262-97da-92ec48e105d0-189-1.c000.snappy.parquet,2026-10-27T21:11:02.910Z
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z,/Volumes/workspace/de_ecommerce/raw_files/incoming/orders/initial/part-00000-tid-7866998030968877259-cbfc3878-f618-4262-97da-92ec48e105d0-189-1.c000.snappy.parquet,2026-10-27T21:11:02.910Z


In [0]:
for t in tables:
    print(t, spark.table(f"{catalog}.{schema}.bronze_{t}").count())

orders 45430
order_items 51234
order_payments 47871
order_reviews 45370
